# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR^2 dataset](https://doi.org/10.71728/senscience.y7m0-f273) using the `mlcroissant` library, referencing all entities by their `@id`. It mirrors best practices for working with Croissant schemas and reproducible scientific data workflows.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Print general metadata information
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"License: {dataset.metadata.license}")
print(f"Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Spatial Coverage: {dataset.metadata.spatialCoverage}")

## 2. Data Overview
Review available record sets, fields, and their IDs using their `@id`s. This step helps you identify which entities you may want to analyze.

In [ ]:
# List all available record sets by their @id
print("Available Record Sets and their Fields (@id):\n")

record_sets = dataset.metadata.recordSets

for rs in record_sets:
    print(f"Record Set: {rs['@id']}")
    if 'fields' in rs:
        print("  Fields:")
        for f in rs['fields']:
            print(f"    - {f['@id']} (name: {f.get('name', '<no name>')})")
    else:
        print("  No fields listed.")
    print("")

## 3. Data Extraction
Load all available record sets into pandas DataFrames. You can select a specific record set for further analysis below, referencing its exact `@id`.

In [ ]:
dataframes = {}
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSets]

for record_set_id in record_set_ids:
    try:
        # Load the records for this record set
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded DataFrame for record set: {record_set_id}, shape: {dataframes[record_set_id].shape}")
        else:
            print(f"Warning: No records extracted for record set {record_set_id}")
    except Exception as e:
        print(f"Error loading records for record set {record_set_id}: {e}")

# Show column names for the first loaded record set
if len(dataframes) > 0:
    first_rs = next(iter(dataframes.keys()))
    print('\nColumns of first available record set DataFrame:')
    print(dataframes[first_rs].columns.tolist())
    dataframes[first_rs].head()
else:
    print("No record sets could be loaded into DataFrames.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalization, and grouping. All column references are made with their `@id` as per the schema documentation.

In [ ]:
# Example EDA: Select a numeric field from the most populated DataFrame
# You may need to inspect the printed columns above to choose appropriate @id values here.
if len(dataframes) > 0:
    # Use the first available record set for demonstration
    record_set_id = next(iter(dataframes.keys()))
    df = dataframes[record_set_id]
    
    print(f"Using record set: {record_set_id}")
    print("Available columns:", df.columns.tolist())
    
    # Try to pick a numeric column: select the first numeric one found
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    
    if numeric_field is not None:
        print(f"Numeric field selected (by @id): {numeric_field}")
        threshold = df[numeric_field].mean()  # Example threshold: mean
        filtered_df = df[df[numeric_field] > threshold]
        print(f"\nFiltered records where {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a categorical/string field (if available)
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == object:
                group_field = col
                break

        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by {group_field} and averaged {numeric_field}:")
            print(grouped_df.head())
        else:
            print("No suitable grouping field found.")
    else:
        print("No numeric field available for analysis.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize the distributions or relationships between fields using their `@id`s. Adjust the fields according to the outputs above.

In [ ]:
# Example visualization: Histogram and boxplot for a numeric field
if len(dataframes) > 0 and numeric_field is not None:
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field} (@id)')
    plt.xlabel(numeric_field)

    plt.subplot(1, 2, 2)
    sns.boxplot(x=df[numeric_field].dropna())
    plt.title(f'Boxplot of {numeric_field} (@id)')

    plt.tight_layout()
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(12, 6))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f'{numeric_field} by {group_field} (@id)')
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 dataset using the `mlcroissant` library, referencing record sets and fields by their `@id` for clarity and reproducibility. We inspected the metadata, previewed several record sets, performed basic EDA, and visualized attributes. For further analysis, consult the dataset documentation and adjust the field `@id` references as needed for your specific research questions.